In [1]:
import pickle
import os
import time
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (classification_report, confusion_matrix, roc_auc_score,
                             roc_curve, precision_recall_curve, auc, accuracy_score)
from scipy.stats import ttest_rel
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (Input, Dense, LSTM, Conv1D, Flatten, concatenate, Dropout,
                                     Multiply, Reshape, BatchNormalization, GlobalAveragePooling1D,
                                     Lambda)
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping

In [4]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense, LSTM, Conv1D, Flatten, concatenate, Dropout, Multiply, Lambda
from tensorflow.keras.layers import BatchNormalization, GlobalAveragePooling1D
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve, precision_recall_curve, auc
import matplotlib.pyplot as plt
import seaborn as sns

In [5]:
import warnings
warnings.filterwarnings("ignore")

In [6]:
df1=pd.read_csv("Wednesday-workingHours.pcap_ISCX.csv")
df2=pd.read_csv("Tuesday-WorkingHours.pcap_ISCX.csv")
df3=pd.read_csv("Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv")
df4=pd.read_csv("Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv")
df5=pd.read_csv("Monday-WorkingHours.pcap_ISCX.csv")
df6=pd.read_csv("Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv")
df7=pd.read_csv("Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv")
df8=pd.read_csv("Friday-WorkingHours-Morning.pcap_ISCX.csv")
df = pd.concat([df1,df2, df3, df4, df5, df6, df7, df8])
del df1
del df2
del df3
del df4
del df5
del df6
del df7
del df8
print(df.shape)

(2830743, 79)


In [7]:
# Drop columns that are completely homogenous

df_nums = df.select_dtypes(include=[np.number])
print("The columns that are completely homogenous are")
for col in df_nums.columns:
    if (df_nums[col].max()-df_nums[col].min() == 0):
        print(col,)
df.drop([col for col in df_nums.columns if df_nums[col].max()-df_nums[col].min() == 0], axis=1, inplace=True)
print(df.shape)

The columns that are completely homogenous are
 Bwd PSH Flags
 Bwd URG Flags
Fwd Avg Bytes/Bulk
 Fwd Avg Packets/Bulk
 Fwd Avg Bulk Rate
 Bwd Avg Bytes/Bulk
 Bwd Avg Packets/Bulk
Bwd Avg Bulk Rate
(2830743, 71)


In [8]:
# Drop NA rows

df.dropna(axis=0, inplace=True)
print(df.shape)

(2829385, 71)


In [9]:
# Normalization
df_nums = df.select_dtypes(include=[np.number])
df_nums = (df_nums-df_nums.min())/(df_nums.max()-df_nums.min())

for col in df_nums.columns:
    df[col] = df_nums[col]

df.dropna(axis=0, inplace=True)
for c in df.columns:
    print(c + " :", df[c].isna().sum())

 Destination Port : 0
 Flow Duration : 0
 Total Fwd Packets : 0
 Total Backward Packets : 0
Total Length of Fwd Packets : 0
 Total Length of Bwd Packets : 0
 Fwd Packet Length Max : 0
 Fwd Packet Length Min : 0
 Fwd Packet Length Mean : 0
 Fwd Packet Length Std : 0
Bwd Packet Length Max : 0
 Bwd Packet Length Min : 0
 Bwd Packet Length Mean : 0
 Bwd Packet Length Std : 0
Flow Bytes/s : 0
 Flow Packets/s : 0
 Flow IAT Mean : 0
 Flow IAT Std : 0
 Flow IAT Max : 0
 Flow IAT Min : 0
Fwd IAT Total : 0
 Fwd IAT Mean : 0
 Fwd IAT Std : 0
 Fwd IAT Max : 0
 Fwd IAT Min : 0
Bwd IAT Total : 0
 Bwd IAT Mean : 0
 Bwd IAT Std : 0
 Bwd IAT Max : 0
 Bwd IAT Min : 0
Fwd PSH Flags : 0
 Fwd URG Flags : 0
 Fwd Header Length : 0
 Bwd Header Length : 0
Fwd Packets/s : 0
 Bwd Packets/s : 0
 Min Packet Length : 0
 Max Packet Length : 0
 Packet Length Mean : 0
 Packet Length Std : 0
 Packet Length Variance : 0
FIN Flag Count : 0
 SYN Flag Count : 0
 RST Flag Count : 0
 PSH Flag Count : 0
 ACK Flag Count : 0
 U

In [10]:
df.shape

(2827876, 71)

In [15]:
# Strip spaces from the column names in df
df.columns = df.columns.str.strip()

X = df.drop(['Label'], axis=1)
y = df['Label'].values


In [17]:
X.shape

(2827876, 70)

In [18]:
# Encode the labels
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)
y_encoded = pd.get_dummies(y_encoded).values  # One-hot encode for multiclass classification

In [21]:
# Define CNN Model
def create_cnn_model(input_shape, num_classes):
    inputs = Input(shape=input_shape)
    x = Conv1D(filters=64, kernel_size=3, activation='relu', padding='same')(inputs)
    x = BatchNormalization()(x)
    x = GlobalAveragePooling1D()(x)
    x = Dense(64, activation='relu')(x)
    x = Dropout(0.3)(x)
    output = Dense(num_classes, activation='softmax')(x)  # Output layer for multiclass classification
    model = Model(inputs, output)
    return model
# Define LSTM Model
def create_lstm_model(input_shape, num_classes):
    inputs = Input(shape=input_shape)
    x = LSTM(64, return_sequences=True)(inputs)
    x = BatchNormalization()(x)
    x = LSTM(32)(x)
    x = Dense(64, activation='relu')(x)
    x = Dropout(0.3)(x)
    output = Dense(num_classes, activation='softmax')(x)  # Output layer for multiclass classification
    model = Model(inputs, output)
    return model
# Define FNN Model
def create_fnn_model(input_shape, num_classes):
    inputs = Input(shape=(input_shape,))
    x = Dense(128, activation='relu')(inputs)
    x = BatchNormalization()(x)
    x = Dense(64, activation='relu')(x)
    x = Dropout(0.3)(x)
    output = Dense(num_classes, activation='softmax')(x)  # Output layer for multiclass classification
    model = Model(inputs, output)
    return model
# Define attention mechanism
def attention_mechanism(inputs):
    attention_weights = Dense(inputs.shape[-1], activation='softmax', name='attention_weights')(inputs)
    attention_output = Multiply(name='attention_output')([inputs, attention_weights])
    return attention_output, attention_weights


In [22]:
from memory_profiler import memory_usage

In [23]:
def train_ensemble():
    return ensemble_model.fit(
        [X_train_cnn, X_train_cnn, X_train], y_train,
        epochs=10,
        batch_size=128,
        validation_split=0.1,
        verbose=0,
        validation_data=([X_test_cnn, X_test_cnn, X_test], y_test)
    )

In [26]:
# Convert one-hot labels back to class labels for StratifiedKFold
y_labels = np.argmax(y_encoded, axis=1)
# --------------------------------- Part 4: 5-Fold Cross-Validation Setup ---------------------------------
kfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

ensemble_accs, cnn_accs, lstm_accs, fnn_accs = [], [], [], []
ensemble_times, cnn_times, lstm_times, fnn_times = [], [], [], []
ensemble_memory_usages, cnn_memory_usages, lstm_memory_usages, fnn_memory_usages = [], [], [], []
all_y_test = []
all_ensemble_pred = []
all_cnn_pred=[]
all_lstm_pred=[]
all_fnn_pred=[]
all_ensemble_prob = []
all_cm_ensemble = []
all_cm_cnn = []
all_cm_lstm = []
all_cm_fnn = []
attention_weights=[]
avg_atts=[]
fold = 1
for train_idx, test_idx in kfold.split(X, y_labels):
    print(f"\n=== Fold {fold} ===")
    fold += 1

    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]

    y_train, y_test = y_encoded[train_idx], y_encoded[test_idx]

    X_train_cnn = X_train.to_numpy().reshape(X_train.shape[0], X_train.shape[1], 1)
    X_test_cnn = X_test.to_numpy().reshape(X_test.shape[0], X_test.shape[1], 1)

    # Number of classes in the dataset
    num_classes = y_train.shape[1]
    # CNN
    cnn_model = create_cnn_model(X_train_cnn.shape[1:], num_classes)
    cnn_model.compile(optimizer=Adam(learning_rate=0.001), loss='categorical_crossentropy', metrics=['accuracy'])
    start = time.time()
    cnn_mem_usage, cnn_history = memory_usage(
    (cnn_model.fit, (X_train_cnn, y_train), {'epochs': 50, 'batch_size': 64, 'validation_split': 0.1, 'verbose': 0}),
    interval=0.1,
    retval=True )
    #cnn_history = cnn_model.fit(X_train_cnn, y_train, epochs=10, batch_size=64, validation_split=0.1, verbose=0)
    end = time.time()
    cnn_times.append(end - start)
    cnn_peak_memory = max(cnn_mem_usage)
    cnn_memory_usages.append(cnn_peak_memory)
    cnn_pred = (cnn_model.predict(X_test_cnn,verbose=0) > 0.5).astype(int)
    cnn_accs.append(accuracy_score(y_test, cnn_pred))
    
    # LSTM
  
    lstm_model = create_lstm_model(X_train_cnn.shape[1:], num_classes)
    lstm_model.compile(optimizer=Adam(learning_rate=0.001), loss='categorical_crossentropy', metrics=['accuracy'])
    start = time.time()
    lstm_mem_usage, lstm_history = memory_usage(
    (lstm_model.fit, (X_train_cnn, y_train), {'epochs': 50, 'batch_size': 64, 'validation_split': 0.1, 'verbose': 0}),
     interval=0.1,retval=True)
    #lstm_history= lstm_model.fit(X_train_cnn, y_train, epochs=10, batch_size=64, validation_split=0.1, verbose=0)
    end = time.time()
    lstm_times.append(end - start)
    lstm_peak_memory = max(lstm_mem_usage)
    lstm_memory_usages.append(lstm_peak_memory)
    lstm_pred = (lstm_model.predict(X_test_cnn,verbose=0) > 0.5).astype(int)
    lstm_accs.append(accuracy_score(y_test, lstm_pred))
    
     # FNN
    
    fnn_model = create_fnn_model(X_train.shape[1], num_classes)
    fnn_model.compile(optimizer=Adam(learning_rate=0.001), loss='categorical_crossentropy', metrics=['accuracy'])
    start = time.time()
    fnn_mem_usage, fnn_history = memory_usage(
    (fnn_model.fit, (X_train, y_train), {'epochs': 50, 'batch_size': 64, 'validation_split': 0.1, 'verbose': 0}),
    interval=0.1,
    retval=True)
    #fnn_history=fnn_model.fit(X_train, y_train, epochs=10, batch_size=64, validation_split=0.1, verbose=0)
    end = time.time()
    fnn_times.append(end - start)
    fnn_peak_memory = max(fnn_mem_usage)
    fnn_memory_usages.append(fnn_peak_memory)
    fnn_pred = (fnn_model.predict(X_test,verbose=0) > 0.5).astype(int)
    fnn_accs.append(accuracy_score(y_test, fnn_pred))
    
     # Ensemble
   
    # Get validation accuracy for dynamic weighting
    cnn_val_accuracy = cnn_model.evaluate(X_test_cnn, y_test, verbose=0)[1]
    lstm_val_accuracy = lstm_model.evaluate(X_test_cnn, y_test, verbose=0)[1]
    fnn_val_accuracy = fnn_model.evaluate(X_test, y_test, verbose=0)[1]
    
    # Calculate dynamic weights based on validation accuracy
    total_accuracy = cnn_val_accuracy + lstm_val_accuracy + fnn_val_accuracy
    weights = {
    'cnn': cnn_val_accuracy / total_accuracy,
    'lstm': lstm_val_accuracy / total_accuracy,
    'fnn': fnn_val_accuracy / total_accuracy
    }
    # Weighted outputs using Lambda layers
    cnn_weighted_output = Lambda(lambda x: x * weights['cnn'])(cnn_model.output)
    lstm_weighted_output = Lambda(lambda x: x * weights['lstm'])(lstm_model.output)
    fnn_weighted_output = Lambda(lambda x: x * weights['fnn'])(fnn_model.output)
    # Combine weighted outputs
    combined = concatenate([cnn_weighted_output, lstm_weighted_output, fnn_weighted_output], name='combined_features')
    # Apply attention mechanism on combined output
    attention_output, attention_tensor = attention_mechanism(combined)
    # Final output layer for multiclass classification
    output = Dense(num_classes, activation='softmax', name='output_layer')(attention_output)
    
    # Directly combine the model outputs without attention for testing
    combined = concatenate([cnn_weighted_output, lstm_weighted_output, fnn_weighted_output], name='combined_features')



   # Define the complete ensemble model
    ensemble_model = Model(inputs=[cnn_model.input, lstm_model.input, fnn_model.input], outputs=output)
    # This returns both the final classification output and attention weights
    ensemble_model_full = Model(inputs=[cnn_model.input, lstm_model.input, fnn_model.input],
                            outputs=[output, attention_tensor])

    # Compile and train the simplified model
    ensemble_model.compile(optimizer=Adam(learning_rate=0.001), loss='categorical_crossentropy', metrics=['accuracy'])
    start = time.time()
    ensemble_mem_usage, ensemble_history = memory_usage(
    train_ensemble,interval=0.1,retval=True)
    #ensemble_history=ensemble_model.fit(
     #   [X_train_cnn, X_train_cnn, X_train], y_train,
       # validation_data=([X_test_cnn, X_test_cnn, X_test], y_test),
        #epochs=50, batch_size=128, verbose=0)
    ensemble_peak_memory = max(ensemble_mem_usage)
    ensemble_memory_usages.append(ensemble_peak_memory)
    end = time.time()
    ensemble_times.append(end - start)
    ensemble_output, att_weights = ensemble_model_full.predict([X_test_cnn, X_test_cnn, X_test], verbose=0)
   # Convert softmax outputs to predicted class indices
    ensemble_pred = np.argmax(ensemble_output, axis=1)
    y_true = np.argmax(y_test, axis=1)

    # Compute accuracy
    ensemble_accs.append(accuracy_score(y_true, ensemble_pred))
    all_y_test.append(y_test)
    all_ensemble_pred.append(ensemble_pred)
    all_ensemble_prob.append(ensemble_model.predict([X_test_cnn, X_test_cnn, X_test],verbose=0))  # Probabilities for ROC/PR
    ensemble_probs, att_weights = ensemble_model_full.predict([X_test_cnn, X_test_cnn, X_test], verbose=0)
    attention_weights.append(att_weights)  # Now att_weights is a real NumPy array
    avg_atts.append(np.mean(att_weights, axis=0))  # Mean over all test samples
    y_pred_ensemble = ensemble_model.predict([X_test_cnn, X_test_cnn, X_test],verbose=0)
    y_pred_classes_ensemble = np.argmax(y_pred_ensemble, axis=1)
    y_true_classes = np.argmax(y_test, axis=1)
    cm_ensemble = confusion_matrix(y_true_classes, y_pred_classes_ensemble)
    all_cm_ensemble.append(cm_ensemble)
    y_pred_classes_cnn = np.argmax(cnn_pred, axis=1)
    cm_cnn=confusion_matrix(y_true_classes,  y_pred_classes_cnn)
    all_cm_cnn.append(cm_cnn)
    y_pred_classes_lstm = np.argmax(lstm_pred, axis=1)
    cm_lstm=confusion_matrix(y_true_classes,y_pred_classes_lstm)
    all_cm_lstm.append(cm_lstm)
    y_pred_classes_fnn = np.argmax(fnn_pred, axis=1)
    cm_fnn=confusion_matrix(y_true_classes,  y_pred_classes_fnn)
    all_cm_fnn.append(cm_fnn)
    # Convert softmax outputs to predicted class indices



=== Fold 1 ===


=== Fold 2 ===

=== Fold 3 ===

=== Fold 4 ===

=== Fold 5 ===


In [36]:

# --------------------------------- Part 5: Report Results ---------------------------------
def report_scores(name, scores, times, memories):
    print(f"{name}: Accuracy = {np.mean(scores):.4f} ± {np.std(scores):.4f}, "
          f"Time = {np.mean(times):.2f}s ± {np.std(times):.2f}s, "
          f"Memory = {np.mean(memories):.2f} MiB ± {np.std(memories):.2f} MiB")

print("\n=== 5-Fold Cross-validation Results ===")
report_scores("CNN", cnn_accs, cnn_times, cnn_memory_usages)
report_scores("LSTM", lstm_accs, lstm_times, lstm_memory_usages)
report_scores("FNN", fnn_accs, fnn_times, fnn_memory_usages)
report_scores("Ensemble", ensemble_accs, ensemble_times, ensemble_memory_usages)


=== 5-Fold Cross-validation Results ===
CNN: Accuracy = 0.8404 ± 0.0088, Time = 9626.37s ± 1512.85s, Memory = 3112.59 MiB ± 1120.89 MiB
LSTM: Accuracy = 0.8710 ± 0.0098, Time = 85834.13s ± 15088.77s, Memory = 2866.91 MiB ± 616.31 MiB
FNN: Accuracy = 0.8704 ± 0.0028, Time = 3639.65s ± 385.24s, Memory = 2445.62 MiB ± 374.70 MiB
Ensemble: Accuracy = 0.8938 ± 0.0002, Time = 17370.50s ± 1722.94s, Memory = 2552.87 MiB ± 421.04 MiB


In [34]:
# --------------------------------- Part 6: Statistical Significance Testing ---------------------------------
print("\n=== Paired t-tests ===")
print("Ensemble vs CNN:", ttest_rel(ensemble_accs, cnn_accs))
print("Ensemble vs LSTM:", ttest_rel(ensemble_accs, lstm_accs))
print("Ensemble vs FNN:", ttest_rel(ensemble_accs, fnn_accs))


=== Paired t-tests ===
Ensemble vs CNN: TtestResult(statistic=12.437116030929486, pvalue=0.00024031608335972255, df=4)
Ensemble vs LSTM: TtestResult(statistic=4.701780554054135, pvalue=0.009296123971838238, df=4)
Ensemble vs FNN: TtestResult(statistic=15.652261452921119, pvalue=9.730081873318588e-05, df=4)


In [40]:
# Compute Classification report for Ensemble Model
from sklearn.metrics import classification_report

# Step 1: Define class labels
class_names = ['BENIGN', 'Bot', 'DDoS', 'DoS GoldenEye', 'DoS Hulk', 'DoS Slowhttptest',
              'DoS slowloris', 'FTP-Patator', 'Heartbleed', 'Infiltration', 'PortScan',
              'SSH-Patator', 'Web Attack- Brute Force', 'Web Attack - Sql Injection', 
              'Web Attack - XSS']

# Step 2: Print classification report
report = classification_report(y_true_classes , y_pred_classes_ensemble, target_names=class_names)
print("Ablation-Classification Report - Ensemble Model-without WGAN-GP and IMOA (5-Fold CV):CICIDS2017-Multiclass Classification")
print(report)



Ablation-Classification Report - Ensemble Model-without WGAN-GP and IMOA (5-Fold CV):CICIDS2017-Multiclass Classification
                            precision    recall  f1-score   support

                    BENIGN       1.00      0.92      0.95   2271320
                       Bot       0.15      0.90      0.26      1956
                      DDoS       0.49      0.94      0.65    128025
             DoS GoldenEye       0.39      0.95      0.55     10293
                  DoS Hulk       0.97      0.78      0.87    230124
          DoS Slowhttptest       0.26      0.98      0.41      5499
             DoS slowloris       0.27      0.97      0.42      5796
               FTP-Patator       0.26      0.89      0.40      7935
                Heartbleed       0.00      0.45      0.00        11
              Infiltration       0.00      0.44      0.01        36
                  PortScan       0.87      0.62      0.72    158804
               SSH-Patator       0.31      0.97      0.47    